        # ⛷️ S3　支線：模型是怎麼學的（手刻梯度下降）
        **統計冒險之旅 2026**　｜　支線（選修，隨時可做）　｜　支線任務　｜　🏅 100 XP

        📖 補充


        ### 🎯 這一關你會學到
        - 損失曲面與下山
- 五行程式的梯度下降
- 學習率太大／太小會怎樣

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "S3"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["S3-1", "S3-2", "S3-3", "S3-4"]
_XP_EACH = 25
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_S3_1(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "亂猜損失"), 26.18507, 0.05): return (False, "亂猜損失 = 損失(0, 0)。")
    return (約等於(抓變數(ns, "真實損失"), 0.14162, 0.02) and bool(抓變數(ns, "真實比較低")), "真實損失 = 損失(3, 2)。")
任務定義("S3-1", _check_S3_1, 提示="損失(3, 2)。")

def _check_S3_2(run):
    out, ns = run()
    if len(抓變數(ns, "歷史")) != 200: return (False, "要走 200 步。")
    if not (約等於(抓變數(ns, "w0"), 2.8537, 0.05) and 約等於(抓變數(ns, "w1"), 2.0954, 0.05)): return (False, "w0、w1 應該接近真實的 (3, 2)。")
    return (bool(抓變數(ns, "一路下降")), "損失應該一路下降。")
任務定義("S3-2", _check_S3_2, 提示="range(200)。")

def _check_S3_3(run):
    out, ns = run()
    if len(抓變數(ns, "歷史大步")) != 30: return (False, "只走 30 步。")
    return (bool(抓變數(ns, "發散")) == True, "步伐太大會跳到山谷另一邊、越跳越高 → 發散。")
任務定義("S3-3", _check_S3_3, 提示="發散 = 歷史大步[-1] > 歷史大步[0]。")

def _check_S3_4(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "公式w1"), 2.09309, 0.01): return (False, "公式w1 = lm.coef_[0]。")
    return (float(抓變數(ns, "差距")) < 0.5, "差距 應該很小：w0、w1 要是 S3-2 走 200 步的結果（若被改掉，重跑 S3-2 那一格）。")
任務定義("S3-4", _check_S3_4, 提示="lm.coef_[0]；w0、w1 是 S3-2 的結果。")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

## ⛷️ 模型是怎麼「學」的？
L04 說線性迴歸「找一條讓誤差平方和最小的直線」。scikit-learn 用公式一次算出來；但更一般的模型（很多參數、沒有公式）是**用走的**：
1. 先隨便猜一組係數（w0, w1）。
2. 算**損失函數**——這組係數離目標多遠（這裡用均方誤差 MSE）。
3. 看腳下的**坡度（梯度）**：往哪個方向改係數，損失會變小。
4. 往低處跨一小步（步伐大小 = **學習率**）。回到 2，直到走不動。

> 🌫️ 起霧的山上看不到路，摸著腳下的坡度往低處走——這就是**梯度下降（gradient descent）**。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
rng = np.random.default_rng(42)
x = rng.uniform(0, 2, 80); y = 3 + 2 * x + rng.normal(0, 0.4, 80)      # 真實規則：y = 3 + 2x
def 損失(w0, w1):
    return np.mean((y - (w0 + w1 * x)) ** 2)
print("亂猜 (0, 0) 的損失", round(損失(0, 0), 3), "| 真實 (3, 2) 的損失", round(損失(3, 2), 3))
# 損失曲面：每一組 (w0, w1) 都有一個高度
W0, W1 = np.meshgrid(np.linspace(-1, 7, 60), np.linspace(-2, 6, 60))
L = np.array([[損失(a, b) for a, b in zip(r0, r1)] for r0, r1 in zip(W0, W1)])
plt.contourf(W0, W1, np.log(L), 30); plt.colorbar(label="log 損失"); plt.xlabel("w0（截距）"); plt.ylabel("w1（斜率）"); plt.title("損失曲面：越暗越低，谷底在 (3, 2) 附近"); plt.show()

## 五行程式的梯度下降
MSE 對 w0、w1 的坡度有公式：`g0 = -2 * mean(y - pred)`、`g1 = -2 * mean((y - pred) * x)`。每一步：`w -= 學習率 * g`。

In [ ]:
w0, w1, 學習率 = 0.0, 0.0, 0.1
歷史 = []
for step in range(200):
    pred = w0 + w1 * x
    g0 = -2 * np.mean(y - pred); g1 = -2 * np.mean((y - pred) * x)
    w0 -= 學習率 * g0; w1 -= 學習率 * g1
    歷史.append(損失(w0, w1))
print("走了 200 步：w0 =", round(w0, 3), "w1 =", round(w1, 3), "損失", round(歷史[-1], 4))
plt.plot(歷史); plt.xlabel("步數"); plt.ylabel("損失"); plt.title("每一步損失都在下降"); plt.show()

### 🎯 任務 S3-1　損失函數

沿用範例的 `x, y, 損失`：算出 `亂猜損失`（w0=0, w1=0）與 `真實損失`（w0=3, w1=2），並把 `真實比較低` 設成布林值。

In [ ]:
# 🎯 任務 S3-1　損失函數（請保留這一行）
亂猜損失 = 損失(0, 0)
真實損失 = ???
真實比較低 = 真實損失 < 亂猜損失
print(round(亂猜損失, 3), round(真實損失, 3), 真實比較低)

In [ ]:
檢查("S3-1")   # ◀ 執行這一格，看看任務 S3-1 有沒有過關

### 🎯 任務 S3-2　走 200 步

從 (0, 0) 出發、學習率 0.1、走 200 步，最後的係數存成 `w0`、`w1`，每一步的損失存成 `歷史`；`一路下降` 是布林值（歷史最後一個是否小於第一個）。

In [ ]:
# 🎯 任務 S3-2　走 200 步（請保留這一行）
w0, w1, 學習率 = 0.0, 0.0, 0.1
歷史 = []
for step in range(???):
    pred = w0 + w1 * x
    g0 = -2 * np.mean(y - pred); g1 = -2 * np.mean((y - pred) * x)
    w0 -= 學習率 * g0; w1 -= 學習率 * g1
    歷史.append(損失(w0, w1))
一路下降 = 歷史[-1] < 歷史[0]
print(round(w0, 3), round(w1, 3), round(歷史[-1], 4), 一路下降)

In [ ]:
檢查("S3-2")   # ◀ 執行這一格，看看任務 S3-2 有沒有過關

### 🎯 任務 S3-3　學習率太大會怎樣？

把學習率改成 1.2、只走 30 步（係數用 `v0`、`v1`，別蓋掉 S3-2 的 `w0`、`w1`），損失存成 `歷史大步`；`發散` 是布林值（最後的損失是否大於第一步的損失）。

In [ ]:
# 🎯 任務 S3-3　學習率太大會怎樣？（請保留這一行）
v0, v1, 學習率 = 0.0, 0.0, 1.2          # 用 v0、v1，別蓋掉 S3-2 走出來的 w0、w1
歷史大步 = []
for step in range(30):
    pred = v0 + v1 * x
    g0 = -2 * np.mean(y - pred); g1 = -2 * np.mean((y - pred) * x)
    v0 -= 學習率 * g0; v1 -= 學習率 * g1
    歷史大步.append(損失(v0, v1))
發散 = ???
print(np.round(歷史大步[:5], 1), "...", f"{歷史大步[-1]:.3g}", 發散)

In [ ]:
檢查("S3-3")   # ◀ 執行這一格，看看任務 S3-3 有沒有過關

### 🎯 任務 S3-4　和 scikit-learn 對答案

用 `LinearRegression` 算出 `公式w0`（截距）與 `公式w1`（斜率），並把和 S3-2 走出來的 `w0`、`w1` 的差距存成 `差距`（兩者絕對差的和）。

In [ ]:
# 🎯 任務 S3-4　和 scikit-learn 對答案（請保留這一行）
lm = LinearRegression().fit(x.reshape(-1, 1), y)
公式w0 = lm.intercept_
公式w1 = ???
差距 = abs(公式w0 - w0) + abs(公式w1 - w1)
print(round(公式w0, 3), round(公式w1, 3), round(差距, 4))

In [ ]:
檢查("S3-4")   # ◀ 執行這一格，看看任務 S3-4 有沒有過關

---
## 🔑 通關密語
　你已經親手讓一個模型「學」了一次——之後看到任何模型的 learning rate、loss，你都知道那是什麼。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**支線完成！** 回入口網頁的「補給站 → 支線任務」蓋章。

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/